# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ibrahim-1rfan/Flyrank-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I am framing this as a Ranking and Scoring task. My goal is not just to classify if a page is decaying, but to assign a priority score to rank the top opportunities. This ensures editorial teams know exactly which articles to rewrite first to maximize traffic recovery.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Filtering for the active mature pages I will be ranking
active_mature = df[(df["content_age_days"] >= 90) & (df["avg_position"] > 0)].copy()

print(f"Total candidate pages to rank: {len(active_mature):,}")


Total candidate pages to rank: 28,795


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I am targeting whether an article is currently experiencing a measurable traffic decline, captured as is_declining_label. This target is derived directly from the observed historical outcome (trend_direction == "down"), meaning the model will learn real-world traffic decay patterns rather than human-defined rules.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The target is an observed historical outcome, not a subjective rule
active_mature["is_declining_label"] = active_mature["trend_direction"].str.lower().eq("down").astype(int)

declining_rate = active_mature["is_declining_label"].mean() * 100
print(f"Base rate of observed decay: {declining_rate:.1f}%")

Base rate of observed decay: 56.4%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric wiill be Precision@K (specifically measuring Top 20 and Top 50). This metric directly calculates the fraction of my top recommended articles that are genuinely decaying. Since content teams have limited bandwidth, "good" means maximizing the true hit rate at the very top of the queue so no writing hours are wasted.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Metric defined: Precision@K")

Metric defined: Precision@K


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one existing, mature content item (URL) belonging to a specific pseudonymized client.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# One row = one existing article (content_id) for a specific client (client_id)
unit_cols = ["content_id", "client_id", "content_type", "content_age_days", "is_declining_label"]

display(active_mature[unit_cols].head(3))

,content_id,client_id,content_type,content_age_days,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed hand rule (like updating all articles older than 180 days with > 500 impressions) works well on a single website but breaks down across different domains. As demonstrated in out-of-sample testing, fixed thresholds fail to generalize to unseen clients. Machine learning captures the tangled, shifting interactions between search position, content age, and click-through rates that simple if-statements cannot handle reliably deep into a ranked list.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# A simple rule threshold degrades because data missingness and feature scales vary by client
rule_candidates = active_mature[
    (active_mature["days_since_last_update"] >= 180) &
    (active_mature["impressions_90d"] >= 500)
]

print(f"A fixed rule rigidly flags {len(rule_candidates):,} pages, ignoring subtle interactions between position and age.")

A fixed rule rigidly flags 17 pages, ignoring subtle interactions between position and age.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.